# Liquidity Hedge Protocol v3 Design and Evaluation

This notebook contains an evaluation of a simplified version of the Liquidity Hedge protocol. Our objective is to evaluate whether the protocol hedge can provide a competitive hedging mechanism for a concentrated liquidity position. Two actor types take opposite sides take engage with the protocol, listed below:

- **Liquidity Position (LP) hedger** (buyer of protection for a given concentrated liquidity position)
- **Rist Taker (RT)underwriter** (seller of protection for a given liquidity position)

The LP hedger buys protection for the decrease of the value of the liquidity position with a defined range. The RT underwriter receives the premium, and commits to paying back after the duration of the protection expires the compensation for the overall position value loss experienced by the LP hedger (whose value decreases non linearly in price change due to Impermanent Loss). In return, the RT actor gains the upside if the position value increases rather than decreasing in the period of coverage, plus a resudual portion of the yield the position provides. The LP's revenue is solely based on the income derived by the liquidity position yield.
Since the protocol must be configured and designed in a way that the price of the premium is anchored to a fair value of the product, and that allows both sides of the protocolto be economically viable over time, we first express and formalize a rigorous design of the protocol, and subsequently we calibrate the parameter configuration to obtain a viable configuration for both actor types. 

This notebook now uses a direct historical backtest (weekly rolling candles over the most recent 3-year real-data window) to evaluate protocol configurations and benchmark hedging strategies. Instead of Monte Carlo selection, strategy break-even thresholds are identified directly from historical weekly outcomes, and protocol configurations are additionally tested for joint LP/RT viability.
## Research questions (RQs)

1. **RQ1 - Pricing mechanism**: What pricing rule is used for the hedge premium, and why is it economically coherent?
2. **RQ2 - Joint viability**: For which protocol fee-split settings does the strategy become jointly viable (LP mean return >= 0 and RT mean return >= 0)?
3. **RQ3 - Relative competitiveness**: How does the protocol compare against two benchmark LP hedging alternatives (fixed hedge perp, dynamic hedge perp) and against unhedged plain LP?
4. **RQ4 - Break-even yield thresholds**: What is the minimum LP daily fee yield required for each strategy to reach non-negative mean LP returns, and how do strategies rank by this threshold?

## Product design constraints used throughout this notebook

These are fixed by design (not optimized):

- **Coverage is full**: `cover = 1`.
- **Barrier equals lower CL bound**: `B = p_l`.
- **CL width is fixed**: `+-7.5%` around weekly entry spot.

These constraints ensure interpretability: only one protocol parameter (`yield_split`) is swept in break-even analysis, while all structural geometry is held fixed.


## 1) Mathematical setup and economic interpretation

This section defines every quantity used later in code.

### 1.1 CL range geometry at weekly entry

Let the weekly entry spot be `S0` and fixed half-width `w = 0.075`.

- Lower bound: `p_l = S0(1 - w)`
- Upper bound: `p_u = S0(1 + w)`
- Barrier: `B = p_l` (by design, not calibrated)

Interpretation: the product protects downside deterioration of LP value inside the corridor logic, with protection anchored at the lower bound.

### 1.2 CL position value and capped corridor payout

Let `V(S)` be the CL mark-to-market value function for the fixed-range position at spot `S`.
Define cap:

$$
Cap = V(S_0) - V(B)
$$

Weekly payout at expiry spot `S_T`:

$$
\Pi(S_T) = \min\left(Cap,\;\max\left(0,\;V(S_0)-V(\max(S_T,B))\right)\right)
$$

Interpretation:

- If price does not create downside loss relative to entry, payout is 0.
- If downside loss exists, payout increases with loss.
- Payout is capped by `Cap`, preventing unbounded RT exposure.

### 1.3 Premium rule used by the protocol

Let:

- `FV` = model-implied expected payout (risk-neutral style approximation via quadrature),
- `m_vol` = volatility-loading multiplier (IV/RV style proxy),
- `y_split` = share of LP fee income transferred to RT,
- `E[Fees]` = expected LP fee income over the week.

Then premium is:

$$
Premium = \max\left(0,\; FV\cdot m_{vol} - y_{split}\cdot E[Fees]\right)
$$

Interpretation: expected LP fees partly subsidize premium (through `y_split`), while `m_vol` inflates fair value to account for risk loading.

### 1.4 Break-even definitions used for ranking

- **Protocol (joint) break-even condition**: both sides must be viable
  - `E[LP return] >= 0`
  - `E[RT return] >= 0`
- **Benchmark LP strategies break-even condition**:
  - `E[LP return] >= 0`

The notebook ranks strategies by the minimum daily LP fee yield (`fee_day`) at which these conditions first hold over the scan grid.


In [1]:
# Cell purpose:
# 1) Import all numerical/data libraries needed by later cells.
# 2) Resolve a robust output folder for generated CSV files.
# 3) Initialize a global random generator for controlled reproducibility.

import os, json, time
import numpy as np
import pandas as pd
import requests
from scipy.special import ndtr
from numpy.polynomial.hermite import hermgauss
import matplotlib.pyplot as plt
from pathlib import Path

# Step-by-step design note:
# We need output files (break-even tables, rankings) to be saved consistently
# even if the notebook is launched from different working directories.
# This resolver searches upward until it finds the repository signature,
# then writes into lh-protocol-v3/notebooks/data.
def resolve_data_dir():
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        repo = parent / 'lh-protocol-v3'
        if (repo / 'protocol-src').exists():
            cand = repo / 'notebooks' / 'data'
            cand.mkdir(parents=True, exist_ok=True)
            return str(cand)
    # Fallback is explicit (still deterministic) if repo signature is absent.
    fallback = here / 'data'
    fallback.mkdir(parents=True, exist_ok=True)
    return str(fallback)

DATA_DIR = resolve_data_dir()

# Fixed seed improves reproducibility of Monte Carlo draws.
# Several simulators additionally build local seeded generators from parameters,
# so repeated runs with same inputs produce stable outputs.
rng = np.random.default_rng(42)
print('DATA_DIR:', DATA_DIR)

# RQ support comment:
# This cell does not answer an RQ directly; it sets controlled infrastructure
# needed to make all later RQ answers auditable and reproducible.


DATA_DIR: /home/sowelo/Scrivania/LiquidityHedge/lh-protocol-v3/notebooks/data


In [2]:
# Cell purpose:
# Download the real daily SOL price series used by all simulations.
# Methodological rule: NO synthetic fallback, NO silent fallback source.
# If Birdeye is unavailable or malformed, stop execution with an explicit error.

# API key is read ONLY from environment variables.
# This avoids hidden hardcoded credentials and keeps execution consistent with project settings.
BIRDEYE_API_KEY = os.getenv('BIRDEYE_API_KEY', '').strip()
SOL_MINT = 'So11111111111111111111111111111111111111112'

def fetch_birdeye_daily_sol(days=1150, chunk_days=90):
    # Step 1: enforce key presence.
    if not BIRDEYE_API_KEY:
        raise RuntimeError('BIRDEYE_API_KEY is missing.')

    # Step 2: define rolling time window (latest ~days observations).
    now = int(time.time())
    start = now - int(days) * 24 * 3600

    # Step 3: pull chunk-by-chunk to avoid oversized requests.
    rows = []
    url = 'https://public-api.birdeye.so/defi/ohlcv'
    headers = {'X-API-KEY': BIRDEYE_API_KEY, 'Accept': 'application/json', 'x-chain': 'solana'}
    cur = start
    while cur < now:
        nxt = min(cur + int(chunk_days) * 24 * 3600, now)
        params = {'address': SOL_MINT, 'type': '1D', 'time_from': int(cur), 'time_to': int(nxt)}
        r = requests.get(url, params=params, headers=headers, timeout=20)
        r.raise_for_status()
        js = r.json()
        if not bool(js.get('success', False)):
            raise RuntimeError(f'Birdeye request failed for [{cur}, {nxt}]')
        items = js.get('data', {}).get('items', [])
        if not isinstance(items, list):
            raise RuntimeError('Unexpected Birdeye schema: data.items is not list')
        rows.extend(items)
        cur = nxt

    # Step 4: schema validation + conversion to canonical dataframe.
    if len(rows) == 0:
        raise RuntimeError('Birdeye returned empty OHLCV items')
    df = pd.DataFrame(rows)

    if 'unixTime' in df.columns:
        ts = pd.to_datetime(df['unixTime'], unit='s', utc=True)
    elif 'startUnixTime' in df.columns:
        ts = pd.to_datetime(df['startUnixTime'], unit='s', utc=True)
    else:
        raise RuntimeError(f'Unexpected Birdeye columns: {list(df.columns)}')

    if 'c' in df.columns:
        close = pd.to_numeric(df['c'], errors='coerce')
    elif 'close' in df.columns:
        close = pd.to_numeric(df['close'], errors='coerce')
    else:
        raise RuntimeError(f'Unexpected Birdeye close columns: {list(df.columns)}')

    out = (
        pd.DataFrame({'open_time': ts, 'close': close})
        .dropna()
        .drop_duplicates('open_time')
        .sort_values('open_time')
        .reset_index(drop=True)
    )

    # Step 5: minimum sample-size guardrail for stability of weekly blocks.
    if len(out) < 365:
        raise RuntimeError(f'Birdeye returned too few daily points ({len(out)})')
    return out

# For 4-year backtests we fetch a wider window than 1460 days to preserve buffer.
hist = fetch_birdeye_daily_sol(days=1700, chunk_days=90)
closes = hist['close'].to_numpy(float)
print('Source: birdeye_SOL_daily')
print('Data points:', len(closes))

# RQ support comment:
# This cell underpins all RQs by ensuring that every result is based on real,
# traceable market data rather than synthetic series.


Source: birdeye_SOL_daily
Data points: 1590


In [3]:
# Cell purpose:
# Define all quantitative primitives and simulation engines used later.
# This is the computational core for RQ1-RQ4.

# -----------------------------
# A) Global modeling constants
# -----------------------------
WIDTH = 0.075                     # Fixed half-width: +-7.5% around S0
T_WEEK = 7/365                    # 1-week horizon in year units
N_LIQ = 1_000.0                   # Liquidity scale factor (normalization)
PROTOCOL_FEE_RATE = 0.010         # Protocol fee retained from premium (1.0%)
QUAD_N = 48                       # Quadrature nodes for expected payoff integral
QUAD_NODES, QUAD_WEIGHTS = hermgauss(QUAD_N)
MARKUP_FLOOR = 0.99               # Lower bound for volatility multiplier

# -----------------------------------------
# B) CL valuation function and position setup
# -----------------------------------------
def cl_value(S, L, p_l, p_u):
    """
    Piecewise CL valuation under fixed range [p_l, p_u].
    Input S can be scalar or vector.
    """
    S = np.asarray(S, float)
    sp, spl, spu = np.sqrt(np.clip(S, 1e-12, None)), np.sqrt(p_l), np.sqrt(p_u)
    below = S <= p_l
    above = S >= p_u
    a = np.where(below, L*(spu-spl)/(spl*spu), np.where(above, 0.0, L*(spu-sp)/(sp*spu)))
    b = np.where(below, 0.0, np.where(above, L*(spu-spl), L*(sp-spl)))
    return a*S + b

def make_pos(S0, lev=1.0):
    """
    Build one weekly position with fixed geometry:
    p_l = S0*(1-WIDTH), p_u = S0*(1+WIDTH), B = p_l.
    Also computes initial value V0 and cap = V0 - V(B).
    """
    p_l, p_u = S0*(1-WIDTH), S0*(1+WIDTH)
    B = p_l
    L = N_LIQ*lev
    V0 = float(cl_value(np.array([S0]), L, p_l, p_u)[0])
    Vb = float(cl_value(np.array([B]), L, p_l, p_u)[0])
    cap = max(0.0, V0 - Vb)
    return {'p_l': p_l, 'p_u': p_u, 'B': B, 'L': L, 'V0': V0, 'cap': cap}

# ----------------------------------
# C) Product payoff and fair-value FV
# ----------------------------------
def corridor_payoff(S_T, S0, pos):
    """Capped downside payout with barrier floor at B = p_l."""
    V_eff = cl_value(np.maximum(S_T, pos['B']), pos['L'], pos['p_l'], pos['p_u'])
    raw = np.maximum(0.0, np.where(S_T >= S0, 0.0, pos['V0'] - V_eff))
    return np.minimum(pos['cap'], raw)

def fv_quadrature(S0, sigma, pos, T=T_WEEK):
    """Expected payout approximation via Gauss-Hermite quadrature."""
    S_T = S0*np.exp(-0.5*sigma**2*T + sigma*np.sqrt(T)*QUAD_NODES*np.sqrt(2))
    pay = corridor_payoff(S_T, S0, pos)
    return max(0.0, float(np.sum(QUAD_WEIGHTS*pay)/np.sqrt(np.pi)))

def iv_rv_regime_switching(s7, s30, stress):
    """
    Simple IV/RV-style loading proxy.
    Maps volatility ratio into [IVRV_MIN, IVRV_MAX] with clipping.
    """
    ratio = float(s7) / max(float(s30), 1e-9)
    IVRV_MIN, IVRV_MAX = 0.99, 1.12
    RATIO_LOW, RATIO_HIGH = 0.90, 1.30
    frac = np.clip((ratio - RATIO_LOW)/(RATIO_HIGH - RATIO_LOW), 0.0, 1.0)
    return float(IVRV_MIN + (IVRV_MAX - IVRV_MIN) * frac)

# Annualized historical volatility anchor from real daily data.
lr_hist = np.diff(np.log(closes))
RV_ANCHOR = float(np.clip(np.std(lr_hist, ddof=1)*np.sqrt(365), 0.10, 3.0))

# -----------------------------------------
# D) Weekly block bootstrap path generation
# -----------------------------------------
def build_weekly_blocks(prices, block_days=8):
    """Create overlapping 8-day blocks to model one-week roll windows."""
    prices = np.asarray(prices, float)
    return np.array([prices[i:i+block_days] for i in range(len(prices)-block_days+1)])

def sample_weekly_path(blocks, n_weeks=52, rng_local=None):
    """Sample n_weeks blocks with replacement."""
    rng_eff = rng_local if rng_local is not None else rng
    idx = rng_eff.integers(0, len(blocks), size=n_weeks)
    return blocks[idx]

blocks = build_weekly_blocks(closes, block_days=8)

# ---------------------------------
# E) Protocol LP+RT Monte Carlo PnL
# ---------------------------------
def simulate_protocol_mc(y_split, fee_day, n_paths=400, n_weeks=52, lev=1.2, tx_bps_open=2.0, tx_bps_settle=2.0):
    """
    Simulate weekly protocol outcomes.
    LP and RT returns are both produced to evaluate joint viability.
    """
    lp_w, rt_w = [], []
    exec_cost_rate = (tx_bps_open + tx_bps_settle)/10_000

    # Deterministic local seed from input parameters for reproducibility.
    seed = int((round(float(y_split),3)*1e6 + round(float(fee_day),7)*1e6)*1000) % 2**32
    rng_local = np.random.default_rng(seed)

    for _ in range(n_paths):
        rt_cap = 2_000_000.0
        idle_yield_week = (1+0.002)**7 - 1  # 0.2% daily idle growth for RT pool capital

        for wk in sample_weekly_path(blocks, n_weeks=n_weeks, rng_local=rng_local):
            S0, ST = float(wk[0]), float(wk[-1])
            lr = np.diff(np.log(wk))
            rv30_proxy = float(np.clip(np.std(lr, ddof=1)*np.sqrt(365), 0.25, 1.8))

            pos = make_pos(S0, lev=lev)
            if pos['cap'] <= 1e-9:
                continue

            # Premium components: fair value, volatility loading, expected fee subsidy.
            fv = fv_quadrature(S0, rv30_proxy, pos)
            ivrv = iv_rv_regime_switching(rv30_proxy, RV_ANCHOR, rv30_proxy/max(RV_ANCHOR,1e-9) > 1.25)
            m_vol = max(MARKUP_FLOOR, ivrv)

            # Approximate expected in-range fee share via normal/log-return proxy.
            t_days = np.arange(1, 8) / 365.0
            sigma = max(rv30_proxy, 1e-9)
            mu = -0.5 * (sigma**2) * t_days
            sd = sigma * np.sqrt(t_days)
            a_log = np.log(1.0 - WIDTH)
            b_log = np.log(1.0 + WIDTH)
            p_in = ndtr((b_log - mu)/sd) - ndtr((a_log - mu)/sd)
            exp_in_range_frac = float(np.mean(p_in))
            exp_fees = pos['V0'] * fee_day * 7 * exp_in_range_frac

            premium = max(0.0, fv*m_vol - y_split*exp_fees)

            # Realized weekly fees use strict in-range fraction from realized path.
            realized_fee_day = float(np.clip(rng_local.normal(fee_day,0.0007),0.0003,0.02))
            in_range_frac_real = float(np.mean((wk[1:] >= pos['p_l']) & (wk[1:] <= pos['p_u'])))
            fees = pos['V0'] * realized_fee_day * 7 * in_range_frac_real

            payoff = float(corridor_payoff(np.array([ST]), S0, pos)[0])
            Vend = float(cl_value(np.array([ST]), pos['L'], pos['p_l'], pos['p_u'])[0])
            protocol_exec_cost = exec_cost_rate * pos['V0']

            lp_pnl = (Vend - pos['V0']) + fees*(1-y_split) + payoff - premium - protocol_exec_cost
            rt_pnl = premium*(1-PROTOCOL_FEE_RATE) + fees*y_split - payoff

            lp_ret = lp_pnl/max(pos['V0'],1e-9)
            rt_ret = rt_pnl/max(rt_cap,1e-9)

            rt_cap *= (1 + rt_ret + idle_yield_week)
            lp_w.append(lp_ret)
            rt_w.append(rt_ret)

    return np.array(lp_w,float), np.array(rt_w,float)

# ---------------------------------------------
# F) Benchmark helper: CL token amount in SOL
# ---------------------------------------------
def cl_amount_sol(S, L, p_l, p_u):
    sp, spl, spu = np.sqrt(max(S,1e-12)), np.sqrt(p_l), np.sqrt(p_u)
    if S <= p_l:
        return L*(spu-spl)/(spl*spu)
    if S >= p_u:
        return 0.0
    return L*(spu-sp)/(sp*spu)

# --------------------------------
# G) Perpetual-hedged LP benchmark
# --------------------------------
def simulate_perp_mc(mode='fixed', fee_day=0.0023, n_paths=400, n_weeks=52, lev=1.2,
                     funding_bps_day=2.0,
                     tx_cost_bps=6.0):
    """
    mode='fixed'   : initial 50% hedge ratio held until week end
    mode='dynamic' : hedge ratio updated when price moves by one decile of range

    Documentable transaction-cost assumption used here:
    - Previous baseline in this notebook was 2 bps per perp trade side.
    - Updated baseline is set to 6 bps per trade side (exactly 3x), matching
      Jupiter support documentation that reports a 0.06% base fee on open/close.

    Why fixed 6 bps (instead of dynamic impact model) in this version:
    - Keeps methodology simple, transparent, and reproducible.
    - Avoids requiring additional venue-state inputs for dynamic impact calibration.

    Source for 0.06% base fee:
    - https://support.jup.ag/hc/en-us/articles/18735867554716-Perpetual-Exchange-Fees
    """
    out=[]
    funding = funding_bps_day/10_000
    tx_cost = tx_cost_bps/10_000

    seed = int((round(float(fee_day),7)*1e6)*1000 + (0 if mode=='fixed' else 1)*999) % 2**32
    rng_local = np.random.default_rng(seed)

    for _ in range(n_paths):
        for wk in sample_weekly_path(blocks, n_weeks=n_weeks, rng_local=rng_local):
            S0, ST = float(wk[0]), float(wk[-1])
            pos = make_pos(S0, lev=lev)

            a0 = cl_amount_sol(S0, pos['L'], pos['p_l'], pos['p_u'])
            hedge_qty = 0.5*a0
            decile = (pos['p_u']-pos['p_l'])/10
            anchor = S0

            fee_income = 0.0
            perp_pnl = 0.0
            funding_cf = 0.0
            tcost = abs(hedge_qty)*S0*tx_cost

            realized_fee_day = float(np.clip(rng_local.normal(fee_day,0.0007),0.0003,0.02))

            for d in range(1,len(wk)):
                s_prev, s_cur = float(wk[d-1]), float(wk[d])
                in_range = (s_prev >= pos['p_l']) and (s_prev <= pos['p_u'])

                # Strict in-range fee accrual baseline: 1.0 in-range, 0.0 out-of-range.
                fee_income += pos['V0'] * realized_fee_day * (1.0 if in_range else 0.0)

                # Short perp offsets long SOL exposure, hence minus sign.
                perp_pnl += -hedge_qty*(s_cur-s_prev)

                # Funding/borrow modeled as explicit daily cost proxy.
                # Jupiter live borrow is hourly and utilization-dependent; fixed proxy retained.
                funding_cf -= abs(hedge_qty*s_prev)*funding

                if mode=='dynamic' and abs(s_cur-anchor)>=decile:
                    drop_frac = max(0.0,(S0-s_cur)/max(pos['p_u']-pos['p_l'],1e-9))
                    target_ratio = float(np.clip(0.5 + drop_frac, 0.1, 1.0))
                    a_cur = cl_amount_sol(s_cur, pos['L'], pos['p_l'], pos['p_u'])
                    target_qty = target_ratio*a_cur
                    dq = target_qty-hedge_qty
                    tcost += abs(dq)*s_cur*tx_cost
                    hedge_qty = target_qty
                    anchor = s_cur

            tcost += abs(hedge_qty)*ST*tx_cost
            Vend = float(cl_value(np.array([ST]), pos['L'], pos['p_l'], pos['p_u'])[0])
            lp_pnl = (Vend-pos['V0']) + fee_income + perp_pnl + funding_cf - tcost
            out.append(lp_pnl/max(pos['V0'],1e-9))

    return np.array(out,float)

# ------------------------
# H) Plain LP benchmark
# ------------------------
def simulate_plain_lp_mc(fee_day=0.0023, n_paths=400, n_weeks=52, lev=1.2):
    out=[]
    seed = int((round(float(fee_day),7)*1e6)*1000) % 2**32
    rng_local = np.random.default_rng(seed)

    for _ in range(n_paths):
        for wk in sample_weekly_path(blocks, n_weeks=n_weeks, rng_local=rng_local):
            S0, ST = float(wk[0]), float(wk[-1])
            pos = make_pos(S0, lev=lev)
            realized_fee_day = float(np.clip(rng_local.normal(fee_day,0.0007),0.0003,0.02))
            in_range_frac_real = float(np.mean((wk[1:] >= pos['p_l']) & (wk[1:] <= pos['p_u'])))
            fees = pos['V0']*realized_fee_day*7*in_range_frac_real
            Vend = float(cl_value(np.array([ST]), pos['L'], pos['p_l'], pos['p_u'])[0])
            out.append(((Vend-pos['V0'])+fees)/max(pos['V0'],1e-9))

    return np.array(out,float)

print('Core simulators ready (strict in-range fee model).')

# RQ support comment:
# - RQ1 is encoded in corridor_payoff + fv_quadrature + premium construction.
# - RQ2 is enabled by simulate_protocol_mc returning both LP and RT returns.
# - RQ3 and RQ4 are enabled by benchmark simulators with aligned cost logic.


Core simulators ready (strict in-range fee model).


In [4]:
# Cell purpose:
# Replace MC evaluation with direct 3-year historical backtest using real daily candles.
# We compute break-even fee_day thresholds for protocol (joint LP+RT viability)
# and benchmark LP strategies, across the requested yield_split values.

# Step 1: requested protocol yield_split candidates.
yield_split_grid = np.array([0.07, 0.08, 0.09, 0.10, 0.11, 0.12, 0.13, 0.14, 0.15, 0.16], dtype=float)
fee_grid_be = np.round(np.linspace(0.0012, 0.0080, 15), 6)

# Step 2: build strict weekly windows from the latest 2 years of real daily candles.
# One week = 8 daily points [day0..day7], rolled every 7 days (weekly open/close cycle).
backtest_days = 1095
if len(closes) < backtest_days + 8:
    raise RuntimeError(f'Insufficient Birdeye history for {backtest_days}-day backtest: {len(closes)} points available')

closes_bt = closes[-(backtest_days + 8):]
weekly_windows = []
for start in range(0, len(closes_bt) - 7, 7):
    wk = closes_bt[start:start+8]
    if len(wk) == 8:
        weekly_windows.append(wk)
weekly_windows = np.array(weekly_windows, dtype=float)

if len(weekly_windows) < 52:
    raise RuntimeError(f'Too few weekly windows for backtest: {len(weekly_windows)}')

# Step 3: backtest simulators (deterministic over historical weekly windows).
def backtest_protocol(y_split, fee_day, lev=1.2, tx_bps_open=2.0, tx_bps_settle=2.0):
    lp_ret, rt_ret = [], []
    exec_cost_rate = (tx_bps_open + tx_bps_settle) / 10_000
    rt_cap = 2_000_000.0
    idle_yield_week = (1 + 0.002) ** 7 - 1

    for wk in weekly_windows:
        S0, ST = float(wk[0]), float(wk[-1])
        pos = make_pos(S0, lev=lev)
        if pos['cap'] <= 1e-9:
            continue

        lr = np.diff(np.log(wk))
        rv30_proxy = float(np.clip(np.std(lr, ddof=1) * np.sqrt(365), 0.25, 1.8))

        fv = fv_quadrature(S0, rv30_proxy, pos)
        ivrv = iv_rv_regime_switching(rv30_proxy, RV_ANCHOR, rv30_proxy / max(RV_ANCHOR, 1e-9) > 1.25)
        m_vol = max(MARKUP_FLOOR, ivrv)

        t_days = np.arange(1, 8) / 365.0
        sigma = max(rv30_proxy, 1e-9)
        mu = -0.5 * (sigma ** 2) * t_days
        sd = sigma * np.sqrt(t_days)
        a_log = np.log(1.0 - WIDTH)
        b_log = np.log(1.0 + WIDTH)
        p_in = ndtr((b_log - mu) / sd) - ndtr((a_log - mu) / sd)
        exp_in_range_frac = float(np.mean(p_in))
        exp_fees = pos['V0'] * fee_day * 7 * exp_in_range_frac

        premium = max(0.0, fv * m_vol - y_split * exp_fees)

        # Backtest uses realized in-range fraction from historical week and fixed candidate fee_day.
        in_range_frac_real = float(np.mean((wk[1:] >= pos['p_l']) & (wk[1:] <= pos['p_u'])))
        fees = pos['V0'] * float(fee_day) * 7 * in_range_frac_real

        payoff = float(corridor_payoff(np.array([ST]), S0, pos)[0])
        Vend = float(cl_value(np.array([ST]), pos['L'], pos['p_l'], pos['p_u'])[0])
        protocol_exec_cost = exec_cost_rate * pos['V0']

        lp_pnl = (Vend - pos['V0']) + fees * (1 - y_split) + payoff - premium - protocol_exec_cost
        rt_pnl = premium * (1 - PROTOCOL_FEE_RATE) + fees * y_split - payoff

        lp_r = lp_pnl / max(pos['V0'], 1e-9)
        rt_r = rt_pnl / max(rt_cap, 1e-9)
        rt_cap *= (1 + rt_r + idle_yield_week)

        lp_ret.append(lp_r)
        rt_ret.append(rt_r)

    return np.array(lp_ret, float), np.array(rt_ret, float)

def backtest_perp(mode='fixed', fee_day=0.0023, lev=1.2, tx_cost_bps=6.0, funding_bps_day=2.0):
    out = []
    tx_cost = tx_cost_bps / 10_000
    funding = funding_bps_day / 10_000

    for wk in weekly_windows:
        S0, ST = float(wk[0]), float(wk[-1])
        pos = make_pos(S0, lev=lev)

        a0 = cl_amount_sol(S0, pos['L'], pos['p_l'], pos['p_u'])
        hedge_qty = 0.5 * a0
        decile = (pos['p_u'] - pos['p_l']) / 10
        anchor = S0

        fee_income = 0.0
        perp_pnl = 0.0
        funding_cf = 0.0
        tcost = abs(hedge_qty) * S0 * tx_cost

        for d in range(1, len(wk)):
            s_prev, s_cur = float(wk[d - 1]), float(wk[d])
            in_range = (s_prev >= pos['p_l']) and (s_prev <= pos['p_u'])
            fee_income += pos['V0'] * float(fee_day) * (1.0 if in_range else 0.0)
            perp_pnl += -hedge_qty * (s_cur - s_prev)
            funding_cf -= abs(hedge_qty * s_prev) * funding

            if mode == 'dynamic' and abs(s_cur - anchor) >= decile:
                drop_frac = max(0.0, (S0 - s_cur) / max(pos['p_u'] - pos['p_l'], 1e-9))
                target_ratio = float(np.clip(0.5 + drop_frac, 0.1, 1.0))
                a_cur = cl_amount_sol(s_cur, pos['L'], pos['p_l'], pos['p_u'])
                target_qty = target_ratio * a_cur
                dq = target_qty - hedge_qty
                tcost += abs(dq) * s_cur * tx_cost
                hedge_qty = target_qty
                anchor = s_cur

        tcost += abs(hedge_qty) * ST * tx_cost
        Vend = float(cl_value(np.array([ST]), pos['L'], pos['p_l'], pos['p_u'])[0])
        lp_pnl = (Vend - pos['V0']) + fee_income + perp_pnl + funding_cf - tcost
        out.append(lp_pnl / max(pos['V0'], 1e-9))

    return np.array(out, float)

def backtest_plain_lp(fee_day=0.0023, lev=1.2):
    out = []
    for wk in weekly_windows:
        S0, ST = float(wk[0]), float(wk[-1])
        pos = make_pos(S0, lev=lev)
        in_range_frac_real = float(np.mean((wk[1:] >= pos['p_l']) & (wk[1:] <= pos['p_u'])))
        fees = pos['V0'] * float(fee_day) * 7 * in_range_frac_real
        Vend = float(cl_value(np.array([ST]), pos['L'], pos['p_l'], pos['p_u'])[0])
        out.append(((Vend - pos['V0']) + fees) / max(pos['V0'], 1e-9))
    return np.array(out, float)

# Step 4: break-even scanners.
def min_fee_lp_only(sim_fn, fee_grid):
    for fd in fee_grid:
        lp = sim_fn(float(fd))
        if len(lp) == 0:
            continue
        if float(np.mean(lp)) >= 0.0:
            return float(fd), float(np.mean(lp))
    return np.nan, np.nan

proto_rows = []
for ys in yield_split_grid:
    be_joint = np.nan
    lp_mean_be = np.nan
    rt_mean_be = np.nan
    for fd in fee_grid_be:
        lp, rt = backtest_protocol(y_split=float(ys), fee_day=float(fd))
        if len(lp) == 0 or len(rt) == 0:
            continue
        lp_m = float(np.mean(lp))
        rt_m = float(np.mean(rt))
        if lp_m >= 0.0 and rt_m >= 0.0:
            be_joint = float(fd)
            lp_mean_be = lp_m
            rt_mean_be = rt_m
            break
    proto_rows.append({
        'strategy': 'protocol_lp',
        'yield_split': float(ys),
        'breakeven_fee_day': be_joint,
        'joint_viable': bool(~np.isnan(be_joint)),
        'lp_mean_at_breakeven': lp_mean_be,
        'rt_mean_at_breakeven': rt_mean_be,
        'weeks': int(len(weekly_windows)),
    })

proto_eval = pd.DataFrame(proto_rows)

plain_be, plain_lp_mean = min_fee_lp_only(lambda fd: backtest_plain_lp(fee_day=fd), fee_grid_be)
perp_fixed_be, perp_fixed_lp_mean = min_fee_lp_only(lambda fd: backtest_perp(mode='fixed', fee_day=fd), fee_grid_be)
perp_dynamic_be, perp_dynamic_lp_mean = min_fee_lp_only(lambda fd: backtest_perp(mode='dynamic', fee_day=fd), fee_grid_be)

bench_eval = pd.DataFrame([
    {'strategy': 'plain_lp', 'yield_split': np.nan, 'breakeven_fee_day': plain_be, 'joint_viable': np.nan, 'lp_mean_at_breakeven': plain_lp_mean, 'rt_mean_at_breakeven': np.nan, 'weeks': int(len(weekly_windows))},
    {'strategy': 'perp_fixed_lp', 'yield_split': np.nan, 'breakeven_fee_day': perp_fixed_be, 'joint_viable': np.nan, 'lp_mean_at_breakeven': perp_fixed_lp_mean, 'rt_mean_at_breakeven': np.nan, 'weeks': int(len(weekly_windows))},
    {'strategy': 'perp_dynamic_lp', 'yield_split': np.nan, 'breakeven_fee_day': perp_dynamic_be, 'joint_viable': np.nan, 'lp_mean_at_breakeven': perp_dynamic_lp_mean, 'rt_mean_at_breakeven': np.nan, 'weeks': int(len(weekly_windows))},
])

rank_all = pd.concat([
    proto_eval[['strategy', 'yield_split', 'breakeven_fee_day', 'joint_viable']],
    bench_eval[['strategy', 'yield_split', 'breakeven_fee_day', 'joint_viable']]
], ignore_index=True).sort_values(['breakeven_fee_day', 'strategy', 'yield_split'], ascending=[True, True, True], na_position='last').reset_index(drop=True)
rank_all['rank'] = np.arange(1, len(rank_all) + 1)

proto_joint_viable_rank = proto_eval[proto_eval['joint_viable']].sort_values(['breakeven_fee_day', 'yield_split'], ascending=[True, True]).reset_index(drop=True)
if len(proto_joint_viable_rank) > 0:
    proto_joint_viable_rank['joint_rank'] = np.arange(1, len(proto_joint_viable_rank) + 1)

# Step 5: save outputs.
proto_eval.to_csv(os.path.join(DATA_DIR, 'v3_paper_protocol_backtest_joint_viability.csv'), index=False)
bench_eval.to_csv(os.path.join(DATA_DIR, 'v3_paper_backtest_benchmark_breakeven.csv'), index=False)
rank_all.to_csv(os.path.join(DATA_DIR, 'v3_paper_backtest_breakeven_ranking_all_strategies.csv'), index=False)
if len(proto_joint_viable_rank) > 0:
    proto_joint_viable_rank.to_csv(os.path.join(DATA_DIR, 'v3_paper_backtest_protocol_joint_viable_ranking.csv'), index=False)

# Step 6: print auditable outputs.
print('Backtest source: Birdeye daily close candles (strict real data)')
print('Weekly windows used:', len(weekly_windows))
print('Yield-split candidates:', list(yield_split_grid))

print('\n--- Protocol (3y backtest): joint viability + break-even ---')
print(proto_eval.to_string(index=False))
print('\n--- Benchmarks (3y backtest): break-even ---')
print(bench_eval.to_string(index=False))
print('\n--- Global ranking (3y backtest, lower break-even is better) ---')
print(rank_all.to_string(index=False))

if len(proto_joint_viable_rank) > 0:
    print('\n--- Joint-viable protocol ranking (3y backtest) ---')
    print(proto_joint_viable_rank.to_string(index=False))
else:
    print('\nNo jointly viable protocol configuration found for the requested yield_split set.')

print('\nSaved:', os.path.join(DATA_DIR, 'v3_paper_protocol_backtest_joint_viability.csv'))
print('Saved:', os.path.join(DATA_DIR, 'v3_paper_backtest_benchmark_breakeven.csv'))
print('Saved:', os.path.join(DATA_DIR, 'v3_paper_backtest_breakeven_ranking_all_strategies.csv'))
if len(proto_joint_viable_rank) > 0:
    print('Saved:', os.path.join(DATA_DIR, 'v3_paper_backtest_protocol_joint_viable_ranking.csv'))

print('\n=== RQ ANSWERS (3Y HISTORICAL BACKTEST) ===')
print('RQ2: jointly viable protocol configurations are rows with joint_viable=True in proto_eval.')
print('RQ3: strategy comparison is rank_all (protocol configs + benchmarks).')
print('RQ4: break-even thresholds are in rank_all.breakeven_fee_day; lower is better.')

rank_all


Backtest source: Birdeye daily close candles (strict real data)
Weekly windows used: 157
Yield-split candidates: [np.float64(0.07), np.float64(0.08), np.float64(0.09), np.float64(0.1), np.float64(0.11), np.float64(0.12), np.float64(0.13), np.float64(0.14), np.float64(0.15), np.float64(0.16)]

--- Protocol (3y backtest): joint viability + break-even ---
   strategy  yield_split  breakeven_fee_day  joint_viable  lp_mean_at_breakeven  rt_mean_at_breakeven  weeks
protocol_lp         0.07                NaN         False                   NaN                   NaN    157
protocol_lp         0.08                NaN         False                   NaN                   NaN    157
protocol_lp         0.09                NaN         False                   NaN                   NaN    157
protocol_lp         0.10                NaN         False                   NaN                   NaN    157
protocol_lp         0.11                NaN         False                   NaN                   Na

,strategy,yield_split,breakeven_fee_day,joint_viable,rank
0,perp_dynamic_lp,NaN,0.005571,NaN,1
1,perp_fixed_lp,NaN,0.006057,NaN,2
2,plain_lp,NaN,0.006057,NaN,3
3,protocol_lp,0.07,NaN,0.0,4
4,protocol_lp,0.08,NaN,0.0,5
5,protocol_lp,0.09,NaN,0.0,6
6,protocol_lp,0.10,NaN,0.0,7
7,protocol_lp,0.11,NaN,0.0,8
8,protocol_lp,0.12,NaN,0.0,9
9,protocol_lp,0.13,NaN,0.0,10


## 5) How to read outputs for the 3-year historical backtest

### Generated artifacts

The backtest cell saves up to four files:

- `v3_paper_protocol_backtest_joint_viability.csv` (protocol rows for requested `yield_split` set with break-even and joint-viability)
- `v3_paper_backtest_benchmark_breakeven.csv` (plain LP and perp benchmark break-even thresholds)
- `v3_paper_backtest_breakeven_ranking_all_strategies.csv` (global ranking across protocol configs and benchmarks)
- `v3_paper_backtest_protocol_joint_viable_ranking.csv` (protocol-only ranking among jointly viable rows)

### Interpretation

- `joint_viable=True` means both LP and RT have non-negative mean returns at that row's break-even threshold.
- `breakeven_fee_day` is the minimum scanned LP daily fee yield where the condition first becomes true on the 3-year weekly historical backtest.
- Lower `breakeven_fee_day` means stronger robustness (less fee environment required to avoid negative mean return).

### RQ mapping in this workflow

- **RQ1 (pricing mechanism):** Cells 1 and 4.
- **RQ2 (joint viability):** `v3_paper_protocol_backtest_joint_viability.csv`.
- **RQ3 (relative competitiveness):** `v3_paper_backtest_breakeven_ranking_all_strategies.csv`.
- **RQ4 (break-even thresholds):** all strategy thresholds in the backtest ranking CSV.

### Methodological limitations

1. Backtest uses one realized 3-year historical path (weekly rollover windows), so it does not provide distributional uncertainty like MC.
2. Perp fee model is simplified but documentable:
   - Trading fee fixed at **6 bps** per trade side (aligned with Jupiter base fee documentation).
   - Borrow/funding fixed at `2 bp/day` proxy (not live utilization-linked hourly borrow state).
3. Break-even is scan-grid exact on finite `fee_grid_be`; finer grids can shift thresholds slightly.

Documented reference for 0.06% base fee:
- [Jupiter Perps Fees FAQ](https://support.jup.ag/hc/en-us/articles/18735867554716-Perpetual-Exchange-Fees)
